In [ ]:
# Ray Compiled Graphs Developer Guide - Hands-on Walkthrough

## 1. Introduction to Ray Compiled Graphs
# Note: Transition to slides to explain "What is Ray Compiled Graphs?" and "Why Use Ray Compiled Graphs?"
# (Discuss performance benefits and specific use cases like LLM inference.)

# Also note that this requires both torch and ray installed (obviously) but both are prepped already as part of the image 
# for this hands-on guide

In [ ]:
# Step 2: Define and Create Actors with Ray Core
import ray

@ray.remote
class EchoActor:
    def echo(self, msg):
        return msg

# Create two actors
a = EchoActor.remote()
b = EchoActor.remote()

In [ ]:
# Send a message and get a response
msg_ref = a.echo.remote("hello")
msg_ref = b.echo.remote(msg_ref)
print(ray.get(msg_ref))  # Expected output: "hello"

In [ ]:
## 3. Using Ray Ray Compiled Graphs for Performance Optimization
# Note: Transition to slides to explain "How Ray Core traditionally executes tasks" 
# and "Challenges with dynamic control flow" (discuss overheads with serialization and object store).

# Step 3: Define and Execute with Ray DAG API (Classic Ray Core, NON Compiled Graphs)
import ray.dag
import time

In [ ]:
# Define a lazy DAG
with ray.dag.InputNode() as inp:
    intermediate_inp = a.echo.bind(inp)
    dag = b.echo.bind(intermediate_inp)

In [ ]:
# Execute the DAG with inputs
print(ray.get(dag.execute("hello")))
print(ray.get(dag.execute("world")))

In [ ]:
# Time the execution
for _ in range(5):
    start = time.perf_counter()
    ray.get(dag.execute("hello"))
    print("Took", time.perf_counter() - start)

In [ ]:
## 4. Optimizing with Ray Compiled Graphs

# Step 4: Compile and Execute with an accelerated directed acyclic graph (aDAG) using Ray Compiled Graphs and time and compare the difference in exec speed
adag = dag.experimental_compile()

In [ ]:
# Execute the aDAG and measure the time
for _ in range(5):
    start = time.perf_counter()
    ray.get(adag.execute("hello"))
    print("Took", time.perf_counter() - start)

In [ ]:
# Tear down the DAG
adag.teardown()

In [ ]:
## 5. [BONUS #1] Multi-Actor Execution in Ray Compiled Graphs

# Step 5: Executing Across Multiple Actors with Ray Compiled Graphs
# Create multiple actors
N = 3
actors = [EchoActor.remote() for _ in range(N)]


In [ ]:

# Define the Graph with multiple outputs
with ray.dag.InputNode() as inp:
    outputs = [actor.echo.bind(inp) for actor in actors]
    dag = ray.dag.MultiOutputNode(outputs)


In [ ]:

# Compile and execute Graph
adag = dag.experimental_compile()
print(ray.get(adag.execute("hello")))  # Expected: ["hello", "hello", "hello"]


In [ ]:
# Tear down the DAG
adag.teardown()